In [3]:
import hashlib
import pickle
import secrets
import statistics
import time
import matplotlib.pyplot as plt

block = {
    "timestamp": time.time(),
    "transactions": ["Alice sends 10 BTC to Bob"],
    "previous_hash": "0",
    "merkle_root": "example_merkle_root"
}

pickled_block = pickle.dumps(block)

target = 3
nonce = 1
block_found = False

start = time.time()

while block_found == False:

    work = hashlib.sha256(
        str(nonce).encode() + pickled_block
    )

    if work.hexdigest()[:target] == "0" * target:

        end = time.time()
        block_found = True

        print("Difficulty:", target)
        print("Nonce:", nonce)
        print("Proof of Work:", work.hexdigest())
        print("Elapsed Time:", end - start)

    nonce += 1

Difficulty: 3
Nonce: 3542
Proof of Work: 0001daab496e66386b3e479847821b5dced459c9b266c36a9e600561d0f1eaba
Elapsed Time: 0.003972053527832031


In [ ]:
import hashlib
import pickle
import secrets
import time
import statistics


def mine_block(pickled_block, target):
    nonce = 1
    attempts = 0
    block_found = False

    start = time.perf_counter()

    while block_found == False:

        work = hashlib.sha256(
            str(nonce).encode() + pickled_block
        )

        attempts += 1

        if work.hexdigest()[:target] == "0" * target:
            end = time.perf_counter()
            block_found = True

            mining_time = end - start

            return nonce, work.hexdigest(), attempts, mining_time

        nonce += 1

In [ ]:
all_results = []

for difficulty in [2, 3, 4, 5]:

    print(f"\n========== DIFFICULTY {difficulty} ==========")

    results = []

    for i in range(30):

        # Create different data for each block
        block = {
            "timestamp": time.time(),
            "transactions": [f"Transaction for Block {i + 1}"],
            "previous_hash": secrets.token_hex(32),
            "merkle_root": secrets.token_hex(32)
        }

        # Convert block to bytes
        pickled_block = pickle.dumps(block)

        # Mine the block
        nonce, valid_hash, attempts, mining_time = mine_block(
            pickled_block,
            difficulty
        )

        # Store result
        result = {
            "block": i + 1,
            "difficulty": difficulty,
            "nonce": nonce,
            "valid_hash": valid_hash,
            "attempts": attempts,
            "mining_time": mining_time
        }

        results.append(result)
        all_results.append(result)

        # Display result
        print(
            f"Block {i + 1}: "
            f"Hash = {valid_hash}, "
            f"Attempts = {attempts}, "
            f"Time = {mining_time:.6f}s"
        )

    print(f"Total Blocks Mined for Difficulty {difficulty}: {len(results)}")


========== DIFFICULTY 2 ==========
Block 1: Hash = 0067848584d3e4a7453aa5e5866ae38176956d366e921c369299fe97abdf83ab, Attempts = 337, Time = 0.000776s
Block 2: Hash = 00b322eb6d87a757c8f42d3e6b309bbaf1c2664b72f5adf75eb574b79281c212, Attempts = 106, Time = 0.000224s
Block 3: Hash = 00f14c66c25e296b641c1b942845df42cc4978d89769e92578846b2bbebb9914, Attempts = 103, Time = 0.000206s
Block 4: Hash = 00daaa4e3726e7828b1c4058e75c548e0587022626fbb700e254b17c12193bb3, Attempts = 392, Time = 0.000850s
Block 5: Hash = 0053bfa2a65126a5cb495c301c979f32f6fa04194b26d65d09937fbb1df37d3f, Attempts = 94, Time = 0.000205s
Block 6: Hash = 0087562b91a489ec6bde05a55ac9cb66cde4f9492ad422b1a86754ac6c1d7472, Attempts = 27, Time = 0.000066s
Block 7: Hash = 00ef521e946dc29eea084934de6a9021b681336db2fe14934e419643783905df, Attempts = 40, Time = 0.000098s
Block 8: Hash = 005f21c89794b08230ddcce0e10d56c5af5d8050032045e54f111bcda235695d, Attempts = 88, Time = 0.000194s
Block 9: Hash = 00a961611fad44e8db44b6c49b4361c9

In [ ]:
import statistics

summary_results = []

for difficulty in [2, 3, 4, 5]:

    # Get the 30 results for this difficulty
    difficulty_results = [
        r for r in all_results
        if r["difficulty"] == difficulty
    ]

    # Extract mining times and attempts
    mining_times = [
        r["mining_time"] for r in difficulty_results
    ]

    attempts = [
        r["attempts"] for r in difficulty_results
    ]

    # Mining time statistics
    average_time = statistics.mean(mining_times)
    minimum_time = min(mining_times)
    maximum_time = max(mining_times)
    std_time = statistics.stdev(mining_times)

    # Average measured hash attempts
    average_attempts = statistics.mean(attempts)

    # Mining throughput = total hashes / total time
    total_attempts = sum(attempts)
    total_time = sum(mining_times)
    hashes_per_second = total_attempts / total_time

    # Theoretical expected attempts
    theoretical_attempts = 16 ** difficulty

    difference = average_attempts - theoretical_attempts

    summary_results.append({
    "difficulty": difficulty,
    "average_time": average_time,
    "minimum_time": minimum_time,
    "maximum_time": maximum_time,
    "std_time": std_time,
    "hashes_per_second": hashes_per_second,
    "average_attempts": average_attempts,
    "theoretical_attempts": theoretical_attempts,
    "difference": difference
})

In [ ]:
print("\nPROOF-OF-WORK MINING PERFORMANCE RESULTS")

print(
    f"{'Difficulty':<12}"
    f"{'Avg Time(s)':<14}"
    f"{'Min Time(s)':<14}"
    f"{'Max Time(s)':<14}"
    f"{'Std Dev(s)':<14}"
    f"{'Hashes/sec':<15}"
    f"{'Measured Avg Attempts':<23}"
    f"{'Theoretical':<16}"
    f"{'Difference':<15}"
)

for r in summary_results:

    difference = (
        r["average_attempts"] - r["theoretical_attempts"]
    )

    print(
        f"{r['difficulty']:<12}"
        f"{r['average_time']:<14.6f}"
        f"{r['minimum_time']:<14.6f}"
        f"{r['maximum_time']:<14.6f}"
        f"{r['std_time']:<14.6f}"
        f"{r['hashes_per_second']:<15.2f}"
        f"{r['average_attempts']:<16.2f}"
        f"{r['theoretical_attempts']:<16}"
        f"{difference:<+15.2f}"
    )


PROOF-OF-WORK MINING PERFORMANCE RESULTS
Difficulty  Avg Time(s)   Min Time(s)   Max Time(s)   Std Dev(s)    Hashes/sec     Measured Avg Attempts  Theoretical     Difference     
2           0.000383      0.000030      0.001585      0.000394      451426.04      173.00          256             -83.00         
3           0.010637      0.000907      0.050026      0.010105      438201.45      4661.20         4096            +565.20        
4           0.112312      0.009386      0.300869      0.088167      438641.13      49264.87        65536           -16271.13      
5           3.787906      0.041363      19.798622     4.543907      389038.49      1473641.40      1048576         +425065.40     
